# Phase 4: LoRA / QLoRA Fine-tuning to Beat RESTBERTa-PM (Acc@1 = 0.8529)

**Target**: Beat RESTBERTa-PM baseline (Acc@1 = 0.8529) on Parameter Matching task.

| Approach | Model | Method | Expected Acc@1 |
|---|---|---|---|
| **A (primary)** | DeBERTa-v3-large (435M) | LoRA r=16 | 0.86 - 0.89 |
| **B (secondary)** | Qwen2.5-3B-Instruct | QLoRA 4-bit r=32 | 0.82 - 0.87 |

**Resumable**: All checkpoints + results saved to Google Drive. Re-run safely — completed steps are skipped.

**GPU**: Optimized for T4 16GB (~8-15 compute units total).

In [ ]:
# Cell 1: Mount Drive + Install dependencies
from google.colab import drive
import os

# Check if Drive is already mounted
if not os.path.exists('/content/drive/MyDrive'):
    print("Mounting Google Drive...")
    drive.mount("/content/drive", force_remount=True)
else:
    print("Google Drive already mounted.")

!pip install -q peft>=0.13.0 bitsandbytes>=0.43.0 accelerate>=0.34.0 trl>=0.12.0
!pip install -q transformers>=4.45.0 datasets

Google Drive already mounted.


In [ ]:
# Cell 2: Download data to Drive (one-time, ~2 min)
import os, shutil

DRIVE_BASE = "/content/drive/MyDrive/restberta"
DRIVE_DATA = os.path.join(DRIVE_BASE, "restberta-datasets")
os.makedirs(DRIVE_BASE, exist_ok=True)

if not os.path.exists(os.path.join(DRIVE_DATA, "parameter_matching/train/train.json")):
    print("Downloading datasets to Drive...")
    os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USERNAME"   # <-- EDIT
    os.environ["KAGGLE_KEY"]      = "YOUR_KAGGLE_KEY"        # <-- EDIT
    !pip install -q kaggle
    !kaggle datasets download -d kailram/restberta-datasets -p /tmp/restberta --unzip
    shutil.copytree("/tmp/restberta", DRIVE_DATA, dirs_exist_ok=True)
    !rm -rf /tmp/restberta
    print("Done.")
else:
    print("Datasets already on Drive.")

# Verify
for split in ["parameter_matching/train", "parameter_matching/validation",
               "endpoint_discovery/train", "endpoint_discovery/validation"]:
    p = os.path.join(DRIVE_DATA, split)
    n = len(os.listdir(p)) if os.path.isdir(p) else 0
    print(f"  {'OK' if n else 'MISSING':>7}  {split}/ ({n} files)")

Datasets already on Drive.
       OK  parameter_matching/train/ (1 files)
       OK  parameter_matching/validation/ (1 files)
       OK  endpoint_discovery/train/ (1 files)
       OK  endpoint_discovery/validation/ (1 files)


In [7]:
# Cell 3: Imports + Config
import os, json, gc, glob, time, random, re, psutil
import numpy as np
import torch
from datetime import datetime
from tqdm import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ── Paths ──
DRIVE_BASE  = "/content/drive/MyDrive/restberta"
DRIVE_DATA  = os.path.join(DRIVE_BASE, "restberta-datasets")
PM_PATH     = os.path.join(DRIVE_DATA, "parameter_matching")
ED_PATH     = os.path.join(DRIVE_DATA, "endpoint_discovery")
RESULTS_DIR = os.path.join(DRIVE_BASE, "results_phase4")
CKPT_DIR    = os.path.join(DRIVE_BASE, "checkpoints_phase4")
CACHE_DIR   = os.path.join(DRIVE_BASE, "cache_phase4")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(CACHE_DIR,   exist_ok=True)

# ── Hyperparameters (AGGRESSIVE RAM savings for 12.7 GB Colab) ──
MAX_LENGTH     = 512
DOC_STRIDE     = 128
N_BEST         = 20
TOP_K          = list(range(1, 11))
SEED           = 42
MAX_TRAIN_QA   = 40_000   # 40K — no overflow = exactly 40K features
EVAL_CHUNK_QA  = 1000     # small eval chunks
EVAL_INF_BATCH = 32       # small inference batch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def print_ram(tag=''):
    r = psutil.virtual_memory()
    g = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
    print(f'  [{tag}] RAM: {r.used/1e9:.1f}/{r.total/1e9:.1f} GB ({r.percent}%) | GPU: {g:.1f} GB')

def result_exists(name, ds_tag):
    matches = glob.glob(os.path.join(RESULTS_DIR, f'{name}_{ds_tag}*.json'))
    return matches[0] if matches else None

def load_existing_result(path):
    with open(path) as f: return json.load(f)

def save_result(result, name, ds_tag):
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    path = os.path.join(RESULTS_DIR, f'{name}_{ds_tag}_{ts}.json')
    with open(path, 'w') as f: json.dump(result, f, indent=2)
    print(f'  Saved: {path}')
    return path

print_ram('init')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Config: {MAX_TRAIN_QA} train samples, no overflow (1 feature per QA pair)')

  [init] RAM: 4.5/13.6 GB (35.8%) | GPU: 0.0 GB
GPU: Tesla T4
Config: 40000 train samples, no overflow (1 feature per QA pair)


### Shared Functions: Data Loading + Evaluation Pipeline
Same evaluation logic as Phase 1/2/3 for fair comparison.

In [8]:
# Cell 5: Data loading + eval functions

def read_qa_pairs_sampled(data_path, split, max_qa):
    # Reservoir sampling: streams 909K, keeps only max_qa in RAM
    reservoir = []
    count = 0
    for fpath in sorted(glob.glob(os.path.join(data_path, split, '*.json'))):
        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                rec = json.loads(line)
                ctx = rec['context']
                for q in rec['questions']:
                    count += 1
                    if len(reservoir) < max_qa:
                        reservoir.append((ctx, q['question'], q['answers']))
                    else:
                        j = random.randint(0, count - 1)
                        if j < max_qa:
                            reservoir[j] = (ctx, q['question'], q['answers'])
    print(f'  Scanned {count}, kept {len(reservoir)}')
    random.shuffle(reservoir)
    return reservoir

def read_qa_pairs_validation(data_path):
    qa = []
    for fpath in sorted(glob.glob(os.path.join(data_path, 'validation', '*.json'))):
        with open(fpath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                rec = json.loads(line)
                ctx = rec['context']
                for q in rec['questions']:
                    qa.append((ctx, q['question'], q['answers']))
    return qa

def identify_properties(ctx, s, e):
    props, cur, on = [], None, False
    for i in range(e - s):
        idx = s + i; c = ctx[idx]
        if c == ' ':
            if on:
                cur['end_char_index'] = idx
                if cur['start_char_index'] == s:
                    b = s - 1
                    while b >= 0 and ctx[b] != ' ':
                        cur['name'] = ctx[b] + cur['name']; cur['partial'] = True; b -= 1
                props.append(cur); cur = None
            on = False
        else:
            if not on:
                cur = {'name': c, 'partial_name': c, 'length': 1,
                       'partial': False, 'start_char_index': idx, 'end_char_index': None}
            else:
                cur['name'] += c; cur['partial_name'] += c; cur['length'] += 1
            on = True
    if cur:
        cur['end_char_index'] = e
        f = e
        while f < len(ctx) and ctx[f] != ' ':
            cur['name'] += ctx[f]; cur['partial'] = True; f += 1
        if not props:
            b = s - 1
            while b >= 0 and ctx[b] != ' ':
                cur['name'] = ctx[b] + cur['name']; cur['partial'] = True; b -= 1
        props.append(cur)
    return props

def determine_best_property(props):
    if not props: return None
    full = None
    for p in props:
        if not p['partial']:
            if full is None: full = p
            else: return None
    if full: return full
    partial, lconflict = None, False
    for p in props:
        if p['partial']:
            if partial is None: partial = p
            elif p['length'] > partial['length']: partial = p; lconflict = False
            elif p['length'] == partial['length']: lconflict = True
    return None if lconflict else partial

def get_answers(om, cls_idx, ctx, sl, el):
    best_s = np.argsort(sl)[-1:-N_BEST-1:-1].tolist()
    best_e = np.argsort(el)[-1:-N_BEST-1:-1].tolist()
    valid = []
    for si in best_s:
        for ei in best_e:
            if si >= len(om) or ei >= len(om): continue
            if om[si] is None or om[ei] is None: continue
            if ei < si: continue
            bp = determine_best_property(identify_properties(ctx, om[si][0], om[ei][1]))
            if bp is None: continue
            valid.append({'score': float(sl[si]+el[ei]), 'property': bp})
    valid.append({'score': float(sl[cls_idx]+el[cls_idx]), 'property': None})
    return sorted(valid, key=lambda x: x['score'], reverse=True)

print('Functions loaded.')
print_ram('funcs')

Functions loaded.
  [funcs] RAM: 4.5/13.6 GB (35.7%) | GPU: 0.0 GB


In [9]:
# Cell 6: Evaluation pipeline (single-threaded, chunked)

def _tokenize_eval_chunk(chunk_qa, tokenizer):
    # Tokenize WITH overflow for eval (needed for accuracy)
    pad_on_right = tokenizer.padding_side == 'right'
    ctx_idx = 1 if pad_on_right else 0
    dt = np.uint32 if tokenizer.vocab_size > 65535 else np.uint16
    c_ids, c_mask, c_off, c_cls, c_oos, c_ans, c_ctx = [], [], [], [], [], [], []
    for b0 in range(0, len(chunk_qa), 64):
        sub = chunk_qa[b0:b0+64]
        tok = tokenizer(
            [x[1] for x in sub] if pad_on_right else [x[0] for x in sub],
            [x[0] for x in sub] if pad_on_right else [x[1] for x in sub],
            truncation='only_second' if pad_on_right else 'only_first',
            max_length=MAX_LENGTH, stride=DOC_STRIDE,
            return_overflowing_tokens=True, return_offsets_mapping=True, padding='max_length')
        sm = tok.pop('overflow_to_sample_mapping')
        for i, orig in enumerate(sm):
            ctx_text, _, answers = sub[orig]
            seq_ids = tok.sequence_ids(i)
            raw_om = tok['offset_mapping'][i]
            om = [(o if seq_ids[k]==ctx_idx else None) for k,o in enumerate(raw_om)]
            cid = tokenizer.cls_token_id or tokenizer.bos_token_id or 0
            try: cp = tok['input_ids'][i].index(cid)
            except ValueError: cp = 0
            if not answers['answer_start']: oos = True
            else:
                sc=answers['answer_start'][0]; ec=sc+len(answers['text'][0])
                s,e=0,len(om)-1
                while s<len(om) and om[s] is None: s+=1
                while e>=0 and om[e] is None: e-=1
                if s>=len(om) or e<0 or sc<om[s][0] or ec>om[e][1]: oos=True
                else:
                    while s<len(om) and om[s] is not None and sc!=om[s][0]: s+=1
                    while e>0 and om[e] is not None and ec!=om[e][1]: e-=1
                    oos=(s>e)
            c_ids.append(np.array(tok['input_ids'][i],dtype=dt))
            c_mask.append(np.array(tok['attention_mask'][i],dtype=np.uint16))
            off=np.full((MAX_LENGTH,2),65535,dtype=np.uint16)
            for k,o in enumerate(om):
                if o is not None: off[k,0]=min(o[0],65534); off[k,1]=min(o[1],65534)
            c_off.append(off); c_cls.append(cp); c_oos.append(oos)
            c_ans.append(answers['text'][0] if answers['text'] else '')
            c_ctx.append(ctx_text)
        del tok, sm
    return (np.stack(c_ids), np.stack(c_mask), np.stack(c_off),
            np.array([(c,o) for c,o in zip(c_cls,c_oos)], dtype=[('cls',np.int16),('oos',np.bool_)]),
            c_ans, c_ctx)

def run_encoder_eval(model, tokenizer, data_path, ds_name, model_name):
    print(f'\n  EVAL: {model_name} on {ds_name}')
    qa_pairs = read_qa_pairs_validation(data_path)
    print(f'  {len(qa_pairs)} validation QA pairs'); print_ram('eval-start')
    results = [{'rank':k,'correct':0,'correct_answerable':0,'correct_non_answerable':0,
                'total':0,'total_answerable':0,'total_non_answerable':0} for k in TOP_K]
    model.eval(); t0=time.time(); feat_done=0
    for c0 in tqdm(range(0,len(qa_pairs),EVAL_CHUNK_QA), desc='  Eval'):
        ids,mask,off,meta,ans,ctx = _tokenize_eval_chunk(qa_pairs[c0:c0+EVAL_CHUNK_QA], tokenizer)
        W=len(meta)
        sl=np.zeros((W,MAX_LENGTH),dtype=np.float32); el=np.zeros((W,MAX_LENGTH),dtype=np.float32)
        for b0 in range(0,W,EVAL_INF_BATCH):
            be=min(b0+EVAL_INF_BATCH,W)
            bi=torch.from_numpy(ids[b0:be].astype(np.int64)).to(device)
            bm=torch.from_numpy(mask[b0:be].astype(np.int64)).to(device)
            with torch.no_grad(): out=model(input_ids=bi,attention_mask=bm)
            sl[b0:be]=out.start_logits.cpu().float().numpy()
            el[b0:be]=out.end_logits.cpu().float().numpy()
            del bi,bm,out
        for i in range(W):
            oi=off[i]
            om=[None if oi[k,0]==65535 else (int(oi[k,0]),int(oi[k,1])) for k in range(oi.shape[0])]
            answers=get_answers(om,int(meta['cls'][i]),ctx[i],sl[i],el[i])
            oos=bool(meta['oos'][i]); rank=None
            for ri,a in enumerate(answers):
                if oos:
                    if a['property'] is None or a['property']['name']==ans[i]: rank=ri+1; break
                else:
                    if a['property'] is not None and a['property']['name']==ans[i]: rank=ri+1; break
            for kr in results:
                if rank is not None and rank<=kr['rank']:
                    kr['correct']+=1
                    if oos: kr['correct_non_answerable']+=1
                    else: kr['correct_answerable']+=1
                kr['total']+=1
                if oos: kr['total_non_answerable']+=1
                else: kr['total_answerable']+=1
        feat_done+=W; del ids,mask,off,meta,ans,ctx,sl,el; gc.collect()
    for kr in results:
        t=kr['total'];ta=kr['total_answerable'];tn=kr['total_non_answerable']
        kr['accuracy']=kr['correct']/t if t else 0
        kr['accuracy_answerable']=kr['correct_answerable']/ta if ta else 0
        kr['accuracy_non_answerable']=kr['correct_non_answerable']/tn if tn else 0
    elapsed=time.time()-t0
    print(f'\n  {elapsed/60:.1f} min | {feat_done} features | Acc@1={results[0]["accuracy"]:.4f}')
    for r in results:
        print(f'  @{r["rank"]:>2}  {r["accuracy"]:.4f}  ans={r["accuracy_answerable"]:.4f}  non={r["accuracy_non_answerable"]:.4f}')
    return {'model':model_name,'dataset':ds_name,'type':'fine-tuned',
            'total_samples':results[0]['total'],'total_answerable':results[0]['total_answerable'],
            'total_non_answerable':results[0]['total_non_answerable'],
            'elapsed_seconds':elapsed,'results':results}

print('Eval pipeline loaded.'); print_ram('eval-loaded')

Eval pipeline loaded.
  [eval-loaded] RAM: 4.5/13.6 GB (35.7%) | GPU: 0.0 GB


---
## Part A: DeBERTa-v3-large + LoRA (Encoder — Primary Approach)

**Why DeBERTa-v3-large?**
- 435M params (3.5x CodeBERT-base) with disentangled attention
- LoRA keeps trainable params at ~2M (0.5%) — fast training, no overfitting
- Encoder QA head directly predicts answer spans — same eval pipeline as baseline
- Phase 2 DeBERTa-base scored only 0.5348 with full fine-tuning on 50K samples → LoRA on large model + 200K samples should dramatically improve

In [10]:
# Cell 8: TOKENIZE training data → save to Drive → free RAM
# FIX: skip answerable samples where answer got truncated
# FIX: balance answerable vs unanswerable (max 1:1 ratio)

DEBERTA_CKPT = 'microsoft/deberta-v3-large'
DEBERTA_NAME = 'deberta_v3_large_lora'
cache_path = os.path.join(CACHE_DIR, f'{DEBERTA_NAME}_train_v2.npz')  # v2!

if os.path.exists(cache_path):
    sz = os.path.getsize(cache_path)/1e6
    print(f'Cache v2 exists ({sz:.0f} MB). Delete to re-tokenize.')
else:
    print('Step 1: Sample training data...')
    print_ram('before-sample')
    train_data = read_qa_pairs_sampled(PM_PATH, 'train', MAX_TRAIN_QA)
    ed = read_qa_pairs_sampled(ED_PATH, 'train', min(4000, MAX_TRAIN_QA//10))
    train_data.extend(ed); del ed
    random.shuffle(train_data)
    print(f'Total: {len(train_data)} QA pairs')
    print_ram('after-sample')

    print('\nStep 2: Tokenize (NO overflow, SKIP truncated answers)...')
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(DEBERTA_CKPT)
    pad_on_right = tok.padding_side == 'right'
    ctx_tok_idx = 1 if pad_on_right else 0

    feat_ids, feat_masks, feat_starts, feat_ends = [], [], [], []
    stats = {'answerable_kept': 0, 'answerable_truncated': 0, 'unanswerable': 0}

    BATCH = 200
    for b0 in tqdm(range(0, len(train_data), BATCH), desc='  Tokenizing'):
        batch = train_data[b0:b0+BATCH]
        ctxs = [x[0] for x in batch]
        ques = [x[1] for x in batch]
        tokenized = tok(
            ques if pad_on_right else ctxs,
            ctxs if pad_on_right else ques,
            truncation='only_second' if pad_on_right else 'only_first',
            max_length=MAX_LENGTH, padding='max_length',
            return_offsets_mapping=True)
        offsets_all = tokenized.pop('offset_mapping')

        for i in range(len(batch)):
            _, _, answers = batch[i]
            input_ids = tokenized['input_ids'][i]
            seq_ids = tokenized.sequence_ids(i)
            cp = 0

            if not answers['answer_start']:
                stats['unanswerable'] += 1
                feat_ids.append(input_ids)
                feat_masks.append(tokenized['attention_mask'][i])
                feat_starts.append(cp)
                feat_ends.append(cp)
                continue

            sc = answers['answer_start'][0]
            ec = sc + len(answers['text'][0])
            ts = 0
            while ts < MAX_LENGTH and seq_ids[ts] != ctx_tok_idx: ts += 1
            te = MAX_LENGTH - 1
            while te >= 0 and seq_ids[te] != ctx_tok_idx: te -= 1

            if ts >= MAX_LENGTH or te < 0:
                stats['answerable_truncated'] += 1
                continue  # SKIP

            if offsets_all[i][ts][0] > sc or offsets_all[i][te][1] < ec:
                stats['answerable_truncated'] += 1
                continue  # SKIP — answer truncated

            while ts < MAX_LENGTH and offsets_all[i][ts][0] <= sc: ts += 1
            sp = max(ts - 1, 0)
            while te >= 0 and offsets_all[i][te][1] >= ec: te -= 1
            ep = min(te + 1, MAX_LENGTH - 1)

            if sp > ep:
                stats['answerable_truncated'] += 1
                continue

            stats['answerable_kept'] += 1
            feat_ids.append(input_ids)
            feat_masks.append(tokenized['attention_mask'][i])
            feat_starts.append(sp)
            feat_ends.append(ep)

        del tokenized, offsets_all, ctxs, ques, batch

    print(f'\nStats:')
    print(f'  Answerable kept:      {stats["answerable_kept"]}')
    print(f'  Answerable truncated: {stats["answerable_truncated"]} (SKIPPED)')
    print(f'  Unanswerable:         {stats["unanswerable"]}')

    # Balance 1:1
    ans_idx = [i for i in range(len(feat_starts)) if feat_starts[i] != 0]
    non_idx = [i for i in range(len(feat_starts)) if feat_starts[i] == 0]
    print(f'  Before balance: {len(ans_idx)} ans, {len(non_idx)} non')

    max_non = len(ans_idx)
    if len(non_idx) > max_non:
        random.shuffle(non_idx)
        non_idx = non_idx[:max_non]

    keep = sorted(ans_idx + non_idx)
    print(f'  After balance: {len(ans_idx)} ans, {len(non_idx)} non = {len(keep)} total')

    all_ids = np.array([feat_ids[i] for i in keep], dtype=np.int32)
    all_masks = np.array([feat_masks[i] for i in keep], dtype=np.int32)
    all_starts = np.array([feat_starts[i] for i in keep], dtype=np.int64)
    all_ends = np.array([feat_ends[i] for i in keep], dtype=np.int64)

    n_ans = np.sum(all_starts != 0)
    print(f'\n  Shape: {all_ids.shape} | Answerable: {n_ans}/{len(all_starts)} ({100*n_ans/len(all_starts):.0f}%)')
    assert n_ans > 0, 'ERROR: zero answerable!'

    np.savez_compressed(cache_path, ids=all_ids, masks=all_masks,
                        starts=all_starts, ends=all_ends)
    print(f'Saved: {cache_path} ({os.path.getsize(cache_path)/1e6:.0f} MB)')

    del all_ids, all_masks, all_starts, all_ends, feat_ids, feat_masks, feat_starts, feat_ends, train_data, tok
    gc.collect()
    print_ram('after-free')
    print('\n>>> RESTART RUNTIME now, then run Cell 3 → 5 → 6 → 9 <<<')


Step 1: Sample training data...
  [before-sample] RAM: 4.5/13.6 GB (35.9%) | GPU: 0.0 GB
  Scanned 0, kept 0
  Scanned 0, kept 0
Total: 0 QA pairs
  [after-sample] RAM: 4.5/13.6 GB (35.9%) | GPU: 0.0 GB

Step 2: Tokenize (NO overflow, SKIP truncated answers)...


  Tokenizing: 0it [00:00, ?it/s]


Stats:
  Answerable kept:      0
  Answerable truncated: 0 (SKIPPED)
  Unanswerable:         0
  Before balance: 0 ans, 0 non
  After balance: 0 ans, 0 non = 0 total

  Shape: (0,) | Answerable: 0/0 (nan%)



/tmp/ipykernel_4985/3211794968.py:114: RuntimeWarning: invalid value encountered in scalar divide
  print(f'\n  Shape: {all_ids.shape} | Answerable: {n_ans}/{len(all_starts)} ({100*n_ans/len(all_starts):.0f}%)')


AssertionError: ERROR: zero answerable!

In [11]:
import os, glob

base = "/content/drive/MyDrive/restberta/restberta-datasets"
print(f"Base exists: {os.path.exists(base)}")

if os.path.exists(base):
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, '').count(os.sep)
        if depth < 3:
            print(f"  {'  '*depth}{os.path.basename(root)}/  ({len(files)} files)")
else:
    # Search for the data
    print("\nSearching Drive for train.json...")
    for root, dirs, files in os.walk("/content/drive/MyDrive"):
        for f in files:
            if f == "train.json":
                print(f"  FOUND: {os.path.join(root, f)}")


Base exists: False

Searching Drive for train.json...


In [5]:
# Cell 9: Train + save FULL model (not just adapter)
try: print_ram
except NameError: raise SystemExit('Re-run Cell 3 first!')

from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

DEBERTA_CKPT = 'microsoft/deberta-v3-large'
DEBERTA_NAME = 'deberta_v3_large_lora'
merged_path = os.path.join(CKPT_DIR, f'{DEBERTA_NAME}_merged')
cache_path = os.path.join(CACHE_DIR, f'{DEBERTA_NAME}_train_v2.npz')

class NpzDataset(Dataset):
    def __init__(self, path):
        d = np.load(path)
        self.ids = d['ids'].astype(np.int64)
        self.masks = d['masks'].astype(np.int64)
        self.starts = d['starts'].astype(np.int64)
        self.ends = d['ends'].astype(np.int64)
        np.clip(self.starts, 0, self.ids.shape[1]-1, out=self.starts)
        np.clip(self.ends, 0, self.ids.shape[1]-1, out=self.ends)
        n_ans = np.sum(self.starts != 0)
        print(f'  {len(self.ids)} features | answerable: {n_ans} ({100*n_ans/len(self.ids):.0f}%)')
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        return {'input_ids': torch.from_numpy(self.ids[i].copy()),
                'attention_mask': torch.from_numpy(self.masks[i].copy()),
                'start_positions': torch.tensor(self.starts[i]),
                'end_positions': torch.tensor(self.ends[i])}

if os.path.exists(merged_path) and os.path.exists(os.path.join(merged_path, 'model.safetensors')):
    print(f'SKIP: merged model already exists at {merged_path}')
else:
    print_ram('before-train')
    train_ds = NpzDataset(cache_path)
    print_ram('data-loaded')

    model = AutoModelForQuestionAnswering.from_pretrained(DEBERTA_CKPT)

    # Detect LoRA targets
    linear_names = set()
    for name, mod in model.named_modules():
        if hasattr(mod, 'weight') and len(mod.weight.shape) == 2:
            linear_names.add(name.split('.')[-1])
    targets = [n for n in ['query_proj','key_proj','value_proj'] if n in linear_names]
    if not targets: targets = list(linear_names)
    print(f'  LoRA targets: {targets}')

    peft_config = LoraConfig(task_type=TaskType.QUESTION_ANS, r=16, lora_alpha=32,
                             lora_dropout=0.05, target_modules=targets)
    model = get_peft_model(model, peft_config)
    for p in model.parameters():
        if p.requires_grad: p.data = p.data.float()
    model.print_trainable_parameters()

    n_steps = len(train_ds) // (2 * 16)
    print(f'  Steps/epoch: ~{n_steps} | 1 epoch only')

    args = TrainingArguments(
        output_dir=os.path.join(CKPT_DIR, f'{DEBERTA_NAME}_tmp'),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=16,
        num_train_epochs=1,
        learning_rate=3e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.06,
        weight_decay=0.01,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        logging_steps=100,
        save_strategy='no',          # DON'T save checkpoints (saves time)
        report_to='none',
        dataloader_num_workers=0,
        dataloader_pin_memory=False,
        remove_unused_columns=False,
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_ds)

    # Test forward pass
    model.train()
    batch = {k: v.unsqueeze(0).to(device) for k, v in train_ds[0].items()}
    with torch.amp.autocast('cuda', dtype=torch.float16):
        out = model(**batch)
    print(f'  Test loss: {out.loss.item():.4f} — OK!')
    del batch, out

    print('\nTraining (1 epoch)...')
    t0 = time.time()
    trainer.train()
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

    # SAVE FULL MERGED MODEL (not just adapter!)
    print('Merging and saving full model...')
    merged = model.merge_and_unload()
    merged.save_pretrained(merged_path)
    AutoTokenizer.from_pretrained(DEBERTA_CKPT).save_pretrained(merged_path)
    print(f'Full model saved to {merged_path}')

    # Verify
    print(f'  qa_outputs: {merged.qa_outputs.weight.shape}')

    del model, merged, trainer, train_ds
    gc.collect(); torch.cuda.empty_cache()
    print_ram('after-train')


  [before-train] RAM: 3.0/13.6 GB (24.7%) | GPU: 0.0 GB


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/restberta/cache_phase4/deberta_v3_large_lora_train_v2.npz'

In [ ]:
# Cell 10: Evaluate DeBERTa-v3-large + LoRA
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from peft import PeftModel

DEBERTA_CKPT = 'microsoft/deberta-v3-large'
DEBERTA_NAME = 'deberta_v3_large_lora'
deberta_ckpt_path = os.path.join(CKPT_DIR, DEBERTA_NAME)
deberta_pm_done = result_exists(DEBERTA_NAME, 'pm')
deberta_ed_done = result_exists(DEBERTA_NAME, 'ed')

all_results_phase4 = []

if deberta_pm_done and deberta_ed_done:
    print('Loading existing results...')
    all_results_phase4.append(load_existing_result(deberta_pm_done))
    all_results_phase4.append(load_existing_result(deberta_ed_done))
else:
    print('Loading model for eval...')
    base = AutoModelForQuestionAnswering.from_pretrained(DEBERTA_CKPT)
    model = PeftModel.from_pretrained(base, deberta_ckpt_path)
    model = model.merge_and_unload()
    model.to(device).half().eval()
    eval_tok = AutoTokenizer.from_pretrained(deberta_ckpt_path)
    print_ram('eval-model-loaded')

    if not deberta_pm_done:
        r = run_encoder_eval(model, eval_tok, PM_PATH, 'parameter_matching', DEBERTA_NAME)
        save_result(r, DEBERTA_NAME, 'pm'); all_results_phase4.append(r)
    else:
        all_results_phase4.append(load_existing_result(deberta_pm_done))

    if not deberta_ed_done:
        r = run_encoder_eval(model, eval_tok, ED_PATH, 'endpoint_discovery', DEBERTA_NAME)
        save_result(r, DEBERTA_NAME, 'ed'); all_results_phase4.append(r)
    else:
        all_results_phase4.append(load_existing_result(deberta_ed_done))

    del model, base, eval_tok; gc.collect(); torch.cuda.empty_cache()
    print_ram('eval-done')

print(f'Results: {len(all_results_phase4)}')

---
## Part B: Qwen2.5-3B-Instruct + QLoRA (Decoder — Faculty Recommended)

**Why Qwen2.5-3B?**
- Best QA benchmark scores after LoRA fine-tuning (distil labs 2025)
- Fits comfortably on T4 with 4-bit QLoRA (~7GB VRAM)
- Faculty specifically requested LoRA/QLoRA on a decoder LLM

**Approach**: Format extractive QA as instruction-following. Model generates exact span from context.

In [ ]:
# Cell 12: Prepare Qwen QLoRA data (20K samples)
QWEN_CKPT = 'Qwen/Qwen2.5-3B-Instruct'
QWEN_NAME = 'qwen25_3b_qlora'
QWEN_MAX_TRAIN = 20_000
QWEN_EVAL_SUBSET = 2000
qwen_ckpt_path = os.path.join(CKPT_DIR, QWEN_NAME)
qwen_pm_done = result_exists(QWEN_NAME, 'pm')
qwen_ed_done = result_exists(QWEN_NAME, 'ed')

if qwen_pm_done and qwen_ed_done:
    print('SKIP: Qwen results exist.')
else:
    print_ram('qwen-prep-start')
    qwen_train = read_qa_pairs_sampled(PM_PATH, 'train', QWEN_MAX_TRAIN)
    ed = read_qa_pairs_sampled(ED_PATH, 'train', 2000)
    qwen_train.extend(ed); del ed; random.shuffle(qwen_train)

    SYS = ('You are an API documentation expert. Extract the exact matching parameter '
           'or endpoint name from the schema. Return ONLY the exact text. '
           'If no match, return NO_ANSWER.')
    train_chats = []
    for c,q,a in qwen_train:
        ans = a['text'][0] if a['text'] and a['answer_start'] else 'NO_ANSWER'
        train_chats.append([
            {'role':'system','content':SYS},
            {'role':'user','content':f'Schema:\n{c}\n\nDescription: {q}'},
            {'role':'assistant','content':ans}])
    del qwen_train; gc.collect()
    print(f'  {len(train_chats)} chat examples'); print_ram('qwen-data')

In [ ]:
# Cell 13: Train Qwen2.5-3B + QLoRA

if qwen_pm_done and qwen_ed_done:
    print(f"SKIP: {QWEN_NAME} already trained + evaluated.")
elif os.path.exists(os.path.join(qwen_ckpt_path, "adapter_config.json")):
    print(f"SKIP training: checkpoint exists at {qwen_ckpt_path}")
else:
    from trl import SFTTrainer, SFTConfig
    print(f"\n{'='*60}")
    print(f"TRAINING: {QWEN_NAME} (QLoRA 4-bit)")
    print(f"{'='*60}")

    # 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Load model
    qwen_model = AutoModelForCausalLM.from_pretrained(
        QWEN_CKPT,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # LoRA config
    qlora_config = LoraConfig(
        r=32,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules="all-linear",
        task_type="CAUSAL_LM",
    )

    # SFT config
    sft_config = SFTConfig(
        output_dir=os.path.join(CKPT_DIR, f"{QWEN_NAME}_tmp"),
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        weight_decay=0.01,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=50,
        save_strategy="epoch",
        save_total_limit=1,
        report_to="none",
        max_seq_length=1024,
        packing=False,
    )

    # Format dataset for SFTTrainer
    from datasets import Dataset as HFDataset

    def gen():
        for chat in train_chats:
            text = qwen_tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)
            yield {"text": text}

    sft_dataset = HFDataset.from_generator(gen)
    print(f"  SFT dataset: {len(sft_dataset)} examples")

    trainer = SFTTrainer(
        model=qwen_model,
        train_dataset=sft_dataset,
        args=sft_config,
        peft_config=qlora_config,
        processing_class=qwen_tok,
    )

    print("\nStarting QLoRA training...")
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    print(f"\nTraining done in {elapsed/60:.1f} min")

    # Save adapter
    trainer.model.save_pretrained(qwen_ckpt_path)
    qwen_tok.save_pretrained(qwen_ckpt_path)
    print(f"Saved adapter to {qwen_ckpt_path}")

    del qwen_model, trainer, sft_dataset
    gc.collect()
    torch.cuda.empty_cache()

if 'train_chats' in dir():
    del train_chats
gc.collect()

In [ ]:
# Cell 14: Evaluate Qwen2.5-3B + QLoRA

def normalize_answer(text):
    return re.sub(r'\s+', ' ', text.strip().strip('"\'').lower())


def run_qwen_eval(model, tokenizer, data_path, ds_name, model_name, n_eval=QWEN_EVAL_SUBSET):
    # Evaluate decoder model on extractive QA (generation-based).
    print(f"\n{'='*60}")
    print(f"  EVAL: {model_name} on {ds_name} ({n_eval} samples)")
    print(f"{'='*60}")

    qa_pairs = read_qa_pairs(data_path, "validation")
    random.seed(SEED)
    random.shuffle(qa_pairs)
    qa_pairs = qa_pairs[:n_eval]

    SYSTEM_MSG = ("You are an API documentation expert. Given an API schema and a description, "
                  "extract the exact matching parameter or endpoint name from the schema. "
                  "Return ONLY the exact text from the schema. If no match exists, return NO_ANSWER.")

    correct = 0
    correct_ans = 0
    correct_non = 0
    total = 0
    total_ans = 0
    total_non = 0

    model.eval()
    t0 = time.time()

    for qi, (ctx, question, answers) in enumerate(tqdm(qa_pairs, desc=f"  {model_name}")):
        gt_text = answers["text"][0] if answers["text"] and answers["answer_start"] else ""
        is_answerable = bool(answers["answer_start"])

        messages = [
            {"role": "system", "content": SYSTEM_MSG},
            {"role": "user", "content": f"Schema:\n{ctx}\n\nDescription: {question}"},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=64,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tokenizer.pad_token_id,
            )
        # Decode only new tokens
        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        prediction = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        # Check correctness
        pred_norm = normalize_answer(prediction)
        gt_norm = normalize_answer(gt_text)
        is_correct = False

        if is_answerable:
            total_ans += 1
            if gt_norm and (pred_norm == gt_norm or gt_norm in pred_norm or pred_norm in gt_norm):
                is_correct = True
                correct_ans += 1
        else:
            total_non += 1
            if pred_norm in ["no_answer", "no answer", "", "n/a", "none"]:
                is_correct = True
                correct_non += 1

        if is_correct:
            correct += 1
        total += 1

    elapsed = time.time() - t0
    acc = correct / total if total else 0
    acc_ans = correct_ans / total_ans if total_ans else 0
    acc_non = correct_non / total_non if total_non else 0

    print(f"\n  Done: {elapsed:.0f}s | {total} samples")
    print(f"  Acc@1: {acc:.4f} | Answerable: {acc_ans:.4f} | Non-ans: {acc_non:.4f}")

    # Format results compatible with Phase 1-3
    results = [{"rank": 1, "correct": correct, "correct_answerable": correct_ans,
                "correct_non_answerable": correct_non, "total": total,
                "total_answerable": total_ans, "total_non_answerable": total_non,
                "accuracy": acc, "accuracy_answerable": acc_ans,
                "accuracy_non_answerable": acc_non}]
    # Pad remaining K values (decoder only does @1)
    for k in range(2, 11):
        results.append({**results[0], "rank": k})

    return {"model": model_name, "dataset": ds_name, "type": "qlora-zero-shot",
            "total_samples": total, "total_answerable": total_ans,
            "total_non_answerable": total_non,
            "elapsed_seconds": elapsed, "results": results}


# Run evaluation
if qwen_pm_done and qwen_ed_done:
    print(f"Loading existing results for {QWEN_NAME}...")
    all_results_phase4.append(load_existing_result(qwen_pm_done))
    all_results_phase4.append(load_existing_result(qwen_ed_done))
else:
    print(f"Loading {QWEN_NAME} for evaluation...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        QWEN_CKPT, quantization_config=bnb_config,
        device_map="auto", trust_remote_code=True,
    )
    qwen_model = PeftModel.from_pretrained(base_model, qwen_ckpt_path)
    qwen_tok = AutoTokenizer.from_pretrained(qwen_ckpt_path, trust_remote_code=True)
    if qwen_tok.pad_token is None:
        qwen_tok.pad_token = qwen_tok.eos_token
    print("Model loaded.")

    if not qwen_pm_done:
        result = run_qwen_eval(qwen_model, qwen_tok, PM_PATH, "parameter_matching", QWEN_NAME)
        save_result(result, QWEN_NAME, "pm")
        all_results_phase4.append(result)
    else:
        all_results_phase4.append(load_existing_result(qwen_pm_done))

    if not qwen_ed_done:
        result = run_qwen_eval(qwen_model, qwen_tok, ED_PATH, "endpoint_discovery", QWEN_NAME)
        save_result(result, QWEN_NAME, "ed")
        all_results_phase4.append(result)
    else:
        all_results_phase4.append(load_existing_result(qwen_ed_done))

    del qwen_model, base_model, qwen_tok
    gc.collect()
    torch.cuda.empty_cache()

print(f"\nPhase 4 results: {len(all_results_phase4)} evaluations")

---
## Full Comparison: Phase 1 + 2 + 3 + 4

In [ ]:
# Cell 16: Load all prior results + comparison table

# Load Phase 1-3 results from Drive (if available)
prior_results = []
prior_dir = os.path.join(DRIVE_BASE, "results")
if os.path.isdir(prior_dir):
    for f in sorted(glob.glob(os.path.join(prior_dir, "*.json"))):
        try:
            with open(f) as fh:
                data = json.load(fh)
                if "results" in data and isinstance(data["results"], list):
                    prior_results.append(data)
        except Exception:
            pass
    print(f"Loaded {len(prior_results)} prior results from {prior_dir}")

# Combine all
combined = prior_results + all_results_phase4

# Flatten any nested lists
flat = []
for item in combined:
    if isinstance(item, list):
        flat.extend(item)
    elif isinstance(item, dict):
        flat.append(item)
combined = flat

# Print comparison
print(f"\n{'='*110}")
print("FULL MODEL COMPARISON — All Phases")
print(f"{'='*110}")
header = f"{'Model':<28} {'Type':<14} {'Dataset':<22} {'N':>7} {'Acc@1':>7} {'Acc@3':>7} {'Acc@5':>7} {'Acc@10':>7}"
print(header)
print("-" * 110)

for r in combined:
    a = {x["rank"]: x["accuracy"] for x in r["results"]}
    model = r.get("model", "?")
    ds = r.get("dataset", "?")
    typ = r.get("type", "fine-tuned")
    n = r.get("total_samples", 0)
    line = (f"{model:<28} {typ:<14} {ds:<22} {n:>7} "
            f"{a.get(1, 0):>7.4f} {a.get(3, 0):>7.4f} {a.get(5, 0):>7.4f} {a.get(10, 0):>7.4f}")

    # Highlight if beating baseline
    if "deberta_v3_large_lora" in model and "parameter" in ds and a.get(1, 0) > 0.8529:
        line += "  <-- BEATS BASELINE!"
    print(line)

print(f"{'='*110}")

# Save combined
with open(os.path.join(RESULTS_DIR, "phase4_full_comparison.json"), "w") as f:
    json.dump(combined, f, indent=2)
print(f"\nSaved to {RESULTS_DIR}/phase4_full_comparison.json")

# Specific comparison
print(f"\n{'='*60}")
print("KEY COMPARISON: Parameter Matching Acc@1")
print(f"{'='*60}")
pm_results = [(r["model"], {x["rank"]: x["accuracy"] for x in r["results"]}.get(1, 0))
              for r in combined if "parameter" in r.get("dataset", "").lower() or "pm" in r.get("dataset", "").lower()]
pm_results.sort(key=lambda x: x[1], reverse=True)
for rank, (model, acc) in enumerate(pm_results, 1):
    marker = " <<<" if "lora" in model.lower() else ""
    print(f"  {rank}. {model:<30} Acc@1 = {acc:.4f}{marker}")

In [1]:
import glob, os, shutil
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

CKPT_DIR = "/content/drive/MyDrive/restberta/checkpoints_phase4"
merged_path = os.path.join(CKPT_DIR, "deberta_v3_large_lora_merged")

# Delete old broken merged model if it exists
if os.path.exists(merged_path):
    shutil.rmtree(merged_path)
    print(f"Deleted old: {merged_path}")

# Find trainer checkpoint (this has the FULL model with QA head)
tmp_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "deberta_v3_large_lora_tmp/checkpoint-*")))
print(f"Trainer checkpoints: {tmp_ckpts}")

if tmp_ckpts:
    ckpt = tmp_ckpts[-1]
    print(f"\nLoading full model from: {ckpt}")
    model = AutoModelForQuestionAnswering.from_pretrained(ckpt)

    # Verify QA head is present
    print(f"  qa_outputs weight shape: {model.qa_outputs.weight.shape}")
    print(f"  QA head OK!")

    # Save as merged model
    model.save_pretrained(merged_path)
    AutoTokenizer.from_pretrained(ckpt).save_pretrained(merged_path)
    print(f"\nSaved merged model to: {merged_path}")
    del model
else:
    print("ERROR: No trainer checkpoint found! You need to re-run training.")


Trainer checkpoints: []
ERROR: No trainer checkpoint found! You need to re-run training.


In [2]:
import os, glob

base = "/content/drive/MyDrive/restberta"

# Check tokenization cache (this should still exist!)
cache = f"{base}/cache_phase4/deberta_v3_large_lora_train_v2.npz"
print(f"v2 cache: {'EXISTS' if os.path.exists(cache) else 'MISSING'}")
if os.path.exists(cache):
    import numpy as np
    d = np.load(cache)
    print(f"  Features: {len(d['ids'])}")
    print(f"  Answerable: {np.sum(d['starts'] != 0)}")

# Check everything else
for d in ['checkpoints_phase4', 'results_phase4']:
    path = os.path.join(base, d)
    if os.path.isdir(path):
        print(f"\n{d}/:")
        for f in sorted(os.listdir(path)):
            print(f"  {f}")


v2 cache: MISSING
